In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r'./Dataset/extracted_dataset/non-multisegment/non-multisegment_input.csv')

df.head()

In [ ]:
import os

# Path to the folder containing the images
image_folder = r'Dataset\all_images'

# Check if the folder exists
if os.path.exists(image_folder):
    # Get all files in the folder
    image_files = [f[24:-4] for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))]
    
    # Print the number of files found
    print(f"Found {len(image_files)} files in {image_folder} folder")
    
    # Display the first few filenames
    if image_files:
        print("First 5 filenames:")
        for file in image_files[:5]:
            print(file)
else:
    print(f"The folder '{image_folder}' does not exist.")

In [ ]:
# Create a map for Dz values and save as JSON
import json

# Initialize the map for Dz values
dz_map = {}

# Select the columns we need (filename and Dz)
dz_cols = ['filename', 'Dz']

# Check if Dz column exists in the dataframe
if 'Dz' in df.columns:
    for row in df[dz_cols].iterrows():
        filename = row[1].values[0]  # Keep the .fsp extension for consistency
        dz_value = row[1].values[1]
        
        # Only add if Dz value is not NaN
        if not pd.isna(dz_value):
            dz_map[filename[:-4]] = float(dz_value)
    
    # Save the map as JSON
    with open('dz.json', 'w') as f:
        json.dump(dz_map, f, indent=2)
    
    print(f"Created dz.json with {len(dz_map)} entries")
    print(f"Sample entries: {dict(list(dz_map.items())[:5])}")
else:
    print("Warning: 'Dz' column not found in the dataframe")
    print(f"Available columns: {df.columns.tolist()}")



In [ ]:
selected_cols= [
    'filename', # Name of the file containing the fault or subfault data
    'LAT',      # Latitude of the fault or subfault patch
    'LON',      # Longitude of the fault or subfault patch
    'DEP',      # Depth of the fault or subfault patch
    'STRK',     # Strike angle (orientation of the fault relative to North)
    'DIP',      # Dip angle (steepness of the fault plane)
    'RAKE',     # Rake angle (direction of slip)
    'LEN_f',    # Fault length (if known before the event)
    'WID',      # Fault width (if known before the event)
    'Htop',     # Depth to the top of the fault
    'HypX',     # Hypocenter location along the fault's length
    'HypZ',     # Hypocenter location along the fault's width
    'Nx',       # Number of subfaults along strike
    'Nz',       # Number of subfaults along dip
    'Dx',       # Length of each subfault patch
    'Dz',       # Width of each subfault patch
]

In [ ]:
# --- Step 3: Plot distributions if data is loaded ---
if not df.empty:
    # Find which of the selected columns are actually in the DataFrame
    existing_cols = [col for col in selected_cols if col in df.columns]
    
    # Isolate the numerical columns for plotting
    numerical_cols = [col for col in existing_cols if pd.api.types.is_numeric_dtype(df[col])]
    
    print(f"Plotting distributions for: {numerical_cols}")

    # --- Step 4: Loop through and plot each numerical column ---
    for col in numerical_cols:
        # Plot 1: Original Distribution
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1) # Create a subplot for the original distribution
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}', fontsize=14)
        plt.xlabel(col)
        plt.ylabel('Frequency')
        
        plt.grid(True, alpha=0.5)
        plt.tight_layout()
        plt.show()
else:
    print("DataFrame is empty. Cannot generate plots.")


In [ ]:
map = {}

for row in df[selected_cols].iterrows():
    filename = row[1].values[0][:-4]
    if filename not in map:
        map[filename] = row[1].values[1:]

In [ ]:
import numpy as np

# Save the map as a .npy file
np.save(r'Dataset/text_vec.npy', map)
print(f"Saved text_vec.npy with {len(map)} entries")
